# 🚀 Day 5: Guided Lab — Churn + Value Modeling with Explainable ML

## Learning Objectives

By the end of this lab, you will be able to:
- Build a reusable preprocessing + modeling **pipeline** for tabular data
- Train and compare **logistic regression → random forest → gradient boosting** for churn
- Evaluate models using **ROC-AUC, PR-AUC, and lift by deciles**
- Predict **MonthlyCharges** (regression) as a value proxy
- Explain predictions using **SHAP** (global + local)
- Build a **Revenue-at-Risk** ranked call list: $p(\text{churn}) \times \widehat{\text{MonthlyCharges}}$

## 🤝 GenAI Copilot (how to use it in this lab)

Use GenAI to **speed up** your work, especially for:
- creating a robust scikit-learn pipeline (imputation + one-hot encoding + model)
- debugging errors (sklearn API changes, SHAP issues)
- translating evaluation metrics into a short business interpretation

**Do not** use GenAI to fabricate results. Any claim in your write-up must be backed by a computed metric/plot.

### Prompt starters
- “Draft a scikit-learn ColumnTransformer pipeline for mixed numeric/categorical churn data.”
- “Explain why PR-AUC is better than accuracy for imbalanced churn.”
- “Help me interpret this SHAP summary plot in business language.”

---
## Part 0: Setup & Data Load (10 min)

In [ ]:
!pip install -q -U google-genai pandas numpy scikit-learn shap
# Optional: if you want AutoML later
# !pip install -q -U flaml

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, precision_recall_curve
)

import shap
shap.initjs()

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42

In [ ]:
# ── GenAI Setup ────────────────────────────────────────────
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not os.environ.get("GEMINI_API_KEY"):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your GEMINI_API_KEY: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL_ID = "gemini-2.5-flash-lite"

# ── Logging Infrastructure ───────────────────────────────────
PROMPT_LOG = []

def _now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

def log_interaction(role, content, label=None):
    """Log a GenAI interaction."""
    entry = {
        "ts": _now(),
        "role": role,
        "content": content if isinstance(content, str) else json.dumps(content),
        "label": label or "",
    }
    PROMPT_LOG.append(entry)
    return entry

print("✅ GenAI + logging ready.")

### Load the Telco dataset

We’ll load the dataset from a public GitHub mirror. If the download fails (no internet), upload the CSV manually and replace the `url`.

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print(f"✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

### Quick cleaning

- Convert `TotalCharges` to numeric (it sometimes contains blanks)
- Keep `customerID` for final outputs, but do not use it as a feature

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
customer_ids = df["customerID"].copy()

# Targets
y_clf = (df["Churn"] == "Yes").astype(int)
y_reg = df["MonthlyCharges"].astype(float)

# Features (drop customerID and Churn; keep MonthlyCharges for classification)
X = df.drop(columns=["customerID", "Churn"])

print(f"Missing values:\n{X.isna().sum().sort_values(ascending=False).head(5)}")
print(f"\nChurn rate: {y_clf.mean():.1%}")

### Train/test split

- For churn we use a **stratified** split (keeps churn rate similar in train and test).
- We will reuse the same split indices for the regression task.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_clf,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

# Use the same row selection for regression
yreg_train = y_reg.loc[X_train.index]
yreg_test = y_reg.loc[X_test.index]

print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")
print(f"Churn rate — train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

---
## Part 1: Preprocessing Pipeline (20 min)

We’ll build a reusable preprocessing component:

- **Numeric:** median impute → standardize
- **Categorical:** most-frequent impute → one-hot encode

Then we can plug in different models (logit, RF, boosting) without rewriting preprocessing.

In [ ]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols[:8]}...")

In [ ]:
# OneHotEncoder API changed across sklearn versions; handle both.
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)

print("✅ Preprocessor defined.")
preprocess

---
## Part 2: Churn Models + Evaluation (20 min)

### Helper functions

We’ll compute ROC-AUC, PR-AUC, precision/recall at a chosen threshold, and lift by deciles (a manager-friendly ranking metric).

In [ ]:
def evaluate_classifier(name, model, X_te, y_te, threshold=0.5):
    """Evaluate a classifier and return metrics + probabilities."""
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= threshold).astype(int)
    out = {
        "model": name,
        "roc_auc": roc_auc_score(y_te, proba),
        "pr_auc": average_precision_score(y_te, proba),
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
    }
    return out, proba, pred


def lift_by_decile(y_true, y_score, n_bins=10):
    """Compute lift by decile (manager-friendly ranking metric)."""
    tmp = pd.DataFrame({"y": y_true, "score": y_score}).copy()
    tmp["decile"] = pd.qcut(tmp["score"].rank(method="first"), q=n_bins, labels=False) + 1
    overall = tmp["y"].mean()
    table = (
        tmp.groupby("decile")
           .agg(n=("y", "size"), churn_rate=("y", "mean"), avg_score=("score", "mean"))
           .sort_index(ascending=False)
           .reset_index()
    )
    table["lift"] = table["churn_rate"] / overall
    return table, overall


def plot_lift(table, overall_rate, title="Lift by decile"):
    """Plot observed churn rate by decile."""
    plt.figure(figsize=(7, 4))
    plt.plot(table["decile"], table["churn_rate"], marker="o")
    plt.axhline(overall_rate, linestyle="--", color="grey", label=f"Overall ({overall_rate:.1%})")
    plt.gca().invert_xaxis()
    plt.xlabel("Decile (10 = highest risk)")
    plt.ylabel("Observed churn rate")
    plt.title(title)
    plt.legend()
    plt.show()


def eval_regression(name, model, X_te, y_te):
    """Evaluate a regression model."""
    pred = model.predict(X_te)
    out = {
        "model": name,
        "mae": mean_absolute_error(y_te, pred),
        "rmse": mean_squared_error(y_te, pred, squared=False),
        "r2": r2_score(y_te, pred)
    }
    return out, pred

print("✅ Helper functions defined.")

### Model 1 — Logistic Regression (baseline)

Logistic regression is a strong baseline because it is fast, stable, and interpretable. We use `class_weight="balanced"` to handle the churn class imbalance.

In [ ]:
logit = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
])
logit.fit(X_train, y_train)

m_logit, p_logit, pred_logit = evaluate_classifier("LogisticRegression", logit, X_test, y_test)
m_logit

In [ ]:
# Confusion matrix (threshold = 0.5)
cm = confusion_matrix(y_test, pred_logit)
disp = ConfusionMatrixDisplay(cm)
disp.plot(values_format="d")
plt.title("Logistic Regression — Confusion Matrix (0.5 threshold)")
plt.show()

### Model 2 — Random Forest

Random forests often improve performance by capturing nonlinear patterns and interactions.

In [ ]:
rf = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])
rf.fit(X_train, y_train)

m_rf, p_rf, pred_rf = evaluate_classifier("RandomForest", rf, X_test, y_test)
m_rf

### Model 3 — Gradient Boosting

Boosting often performs very well on tabular data.

In [ ]:
gb = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))
])
gb.fit(X_train, y_train)

m_gb, p_gb, pred_gb = evaluate_classifier("GradientBoosting", gb, X_test, y_test)
m_gb

### Compare models

In [ ]:
clf_results = pd.DataFrame([m_logit, m_rf, m_gb]).sort_values(["pr_auc", "roc_auc"], ascending=False)
clf_results

In [ ]:
# ── ROC + PR Curves ───────────────────────────────────────
model_probas = {"LogisticRegression": p_logit, "RandomForest": p_rf, "GradientBoosting": p_gb}
model_metrics = {"LogisticRegression": m_logit, "RandomForest": m_rf, "GradientBoosting": m_gb}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, proba in model_probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax1.plot(fpr, tpr, label=f"{name} (AUC={model_metrics[name]['roc_auc']:.3f})")

ax1.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

for name, proba in model_probas.items():
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ax2.plot(rec, prec, label=f"{name} (AUC={model_metrics[name]['pr_auc']:.3f})")

baseline = y_test.mean()
ax2.axhline(y=baseline, color="k", linestyle="--", alpha=0.3, label=f"Baseline ({baseline:.2f})")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curves")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Lift by deciles (manager-friendly ranking view)

In [ ]:
# Choose the best churn model for the rest of the lab
best_name = clf_results.iloc[0]["model"]
best_model = {"LogisticRegression": logit, "RandomForest": rf, "GradientBoosting": gb}[best_name]
best_proba = {"LogisticRegression": p_logit, "RandomForest": p_rf, "GradientBoosting": p_gb}[best_name]

lift_tbl, overall = lift_by_decile(y_test, best_proba, n_bins=10)
print(f"Best model: {best_name}")
lift_tbl

In [ ]:
plot_lift(lift_tbl, overall, title=f"{best_name} — Lift by decile")

### GenAI: Interpret the model comparison

Let’s ask Gemini to help interpret these results in business language.

In [ ]:
# ── GenAI: Interpret Model Comparison ─────────────────────
metrics_summary = clf_results.to_string(index=False)
lift_top = lift_tbl.head(3).to_string(index=False)

interpret_prompt = f"""Here are churn model results on a holdout test set:

{metrics_summary}

Lift by decile (top 3 deciles of the best model):
{lift_top}

Context: We are building a churn retention campaign. Missing a churner (false negative)
costs roughly 5× more than contacting a non-churner (false positive).

In 3–4 sentences, recommend which model to use and why. Mention the precision-recall
tradeoff and what the lift chart tells a manager about targeting."""

log_interaction("user", interpret_prompt, label="model_interpretation")

response = client.models.generate_content(model=MODEL_ID, contents=interpret_prompt)
interpretation = response.text
log_interaction("assistant", interpretation, label="model_interpretation")

print("🤖 Gemini's Model Recommendation:")
print("=" * 60)
print(interpretation)

---
## Part 3: Regression — Predict MonthlyCharges (15 min)

We’ll predict **MonthlyCharges** using the same feature set.

Important: Do **not** include MonthlyCharges itself as a predictor when forecasting it.

In [ ]:
# Regression features (drop MonthlyCharges from X)
Xr_train = X_train.drop(columns=["MonthlyCharges"])
Xr_test = X_test.drop(columns=["MonthlyCharges"])

numeric_cols_r = Xr_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols_r = Xr_train.select_dtypes(exclude=["number"]).columns.tolist()

preprocess_r = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols_r),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)]), categorical_cols_r),
    ]
)

### Regression models: Ridge → Random Forest → Gradient Boosting

In [ ]:
ridge = Pipeline(steps=[("prep", preprocess_r), ("reg", Ridge(alpha=1.0))])
ridge.fit(Xr_train, yreg_train)
m_ridge, pred_ridge = eval_regression("Ridge", ridge, Xr_test, yreg_test)

rfr = Pipeline(steps=[
    ("prep", preprocess_r),
    ("reg", RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1))
])
rfr.fit(Xr_train, yreg_train)
m_rfr, pred_rfr = eval_regression("RandomForestRegressor", rfr, Xr_test, yreg_test)

gbr = Pipeline(steps=[
    ("prep", preprocess_r),
    ("reg", GradientBoostingRegressor(random_state=RANDOM_STATE))
])
gbr.fit(Xr_train, yreg_train)
m_gbr, pred_gbr = eval_regression("GradientBoostingRegressor", gbr, Xr_test, yreg_test)

reg_results = pd.DataFrame([m_ridge, m_rfr, m_gbr]).sort_values(["mae", "rmse"], ascending=True)
reg_results

---
## Part 4: SHAP + Revenue-at-Risk Call List (10 min)

### SHAP (global + local)

We’ll explain the **best churn model**. To keep SHAP fast, we transform features using the pipeline’s preprocessing and sample a subset of rows.

In [ ]:
# Extract transformed matrices + feature names
prep_fitted = best_model.named_steps["prep"]
clf_fitted = best_model.named_steps["clf"]

X_train_enc = prep_fitted.transform(X_train)
feature_names = prep_fitted.get_feature_names_out()

# Sample for speed
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    X_train_enc.shape[0], size=min(800, X_train_enc.shape[0]), replace=False
)
X_shap = X_train_enc[sample_idx]

print(f"Encoded shape: {X_train_enc.shape}")
print(f"SHAP sample shape: {X_shap.shape}")

In [ ]:
# ── SHAP: Global Summary Plot ───────────────────────────────
explainer = None
try:
    explainer = shap.TreeExplainer(clf_fitted)
    shap_values = explainer.shap_values(X_shap)
except Exception as e:
    print(f"TreeExplainer failed, falling back to shap.Explainer: {e}")
    explainer = shap.Explainer(clf_fitted, X_shap)
    shap_values = explainer(X_shap)

plt.figure(figsize=(10, 6))
try:
    # For some classifiers, shap_values is a list; class 1 is churn
    if isinstance(shap_values, list):
        shap.summary_plot(shap_values[1], X_shap, feature_names=feature_names, max_display=15, show=False)
    else:
        shap.summary_plot(shap_values, X_shap, feature_names=feature_names, max_display=15, show=False)
    plt.title(f"SHAP Summary — {best_name}")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not render summary plot: {e}")

### Local explanation for one customer

In [ ]:
row_i = 0
x_one = X_shap[row_i:row_i+1]

try:
    if hasattr(explainer, "__call__") and not isinstance(shap_values, list):
        exp = explainer(x_one)
        shap.plots.waterfall(exp[0])
    else:
        # TreeExplainer older API
        sv = shap_values[1][row_i] if isinstance(shap_values, list) else shap_values[row_i]
        base = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
        shap.plots._waterfall.waterfall_legacy(base, sv, feature_names=feature_names)
except Exception as e:
    print(f"Local explanation failed: {e}")

### GenAI: Turn SHAP into a narrative

Let’s ask Gemini to interpret the SHAP values for a specific customer in plain business language.

In [ ]:
# ── GenAI: SHAP Narrative ─────────────────────────────────
# Get top SHAP contributors for this customer
if isinstance(shap_values, list):
    sv_row = shap_values[1][row_i]
else:
    sv_row = shap_values[row_i]

shap_df = pd.DataFrame({"feature": feature_names, "shap_value": sv_row})
shap_df["abs_shap"] = shap_df["shap_value"].abs()
top_features = shap_df.nlargest(6, "abs_shap")

shap_summary = "\n".join(
    f"  - {row['feature']}: SHAP={row['shap_value']:+.3f} ({'increases' if row['shap_value'] > 0 else 'decreases'} churn risk)"
    for _, row in top_features.iterrows()
)

narrative_prompt = f"""A customer has been flagged by our churn model.
The overall churn rate is {y_test.mean():.0%}.

Top factors driving this prediction (SHAP values):
{shap_summary}

Write a 3-sentence explanation for a retention manager who needs to decide
whether to call this customer. Use plain business language, not technical jargon."""

log_interaction("user", narrative_prompt, label="shap_narrative")

response = client.models.generate_content(model=MODEL_ID, contents=narrative_prompt)
narrative = response.text
log_interaction("assistant", narrative, label="shap_narrative")

print("🤖 GenAI Narrative for Retention Manager:")
print("=" * 60)
print(narrative)

### Revenue-at-Risk ranking

$$\text{Revenue at Risk} = p(\text{churn}) \times \widehat{\text{MonthlyCharges}}$$

Then create a ranked call list.

In [ ]:
# Choose best regression model
best_reg_name = reg_results.iloc[0]["model"]
best_reg_model = {"Ridge": ridge, "RandomForestRegressor": rfr, "GradientBoostingRegressor": gbr}[best_reg_name]

# Predictions on the test set
p_churn = best_proba
pred_value = best_reg_model.predict(Xr_test)
pred_value = np.clip(pred_value, 0, None)  # avoid negative predictions

call_list = pd.DataFrame({
    "customerID": customer_ids.loc[X_test.index].values,
    "p_churn": p_churn,
    "pred_monthly_charges": pred_value,
})
call_list["revenue_at_risk"] = call_list["p_churn"] * call_list["pred_monthly_charges"]

call_list = call_list.sort_values("revenue_at_risk", ascending=False)
print(f"Top 10 customers by Revenue-at-Risk:")
call_list.head(10)

In [ ]:
# Save top-N call list
TOP_N = 200
out_path = "day5_guided_call_list_top200.csv"
call_list.head(TOP_N).to_csv(out_path, index=False)
print(f"✅ Saved top {TOP_N} call list to {out_path}")

---
## Wrap-up

**What you built:**
- A reproducible ML pipeline (imputation + encoding + model)
- Model progression: logistic regression → random forest → gradient boosting
- Business evaluation: ROC/PR curves + lift by deciles
- Regression for MonthlyCharges as a value proxy
- SHAP explanations (global + local) with GenAI narrative
- Revenue-at-Risk ranked call list

**Next:** In the **Independent Lab**, you’ll choose an extension track (cost targeting, calibration, AutoML, or segment stress test) and turn this into a manager-ready recommendation.

In [ ]:
# ── Export Prompt Log ───────────────────────────────────────
if PROMPT_LOG:
    log_df = pd.DataFrame(PROMPT_LOG)
    log_df.to_csv("day5_lab1_prompt_log.csv", index=False)
    print(f"✅ Exported {len(log_df)} log entries to day5_lab1_prompt_log.csv")
else:
    print("No GenAI interactions logged.")